<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔧 Lab 00: Foundry Project Setup & Prerequisites</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 What We Learn

In this lab, we walk through the **one-time setup** required before starting the workshop. We will create a **Microsoft Foundry project**, deploy the **AI models** our agents will use, enable **Application Insights** for tracing and observability, and configure the **environment variables** that all subsequent labs depend on.

---


## 🧳 The Contoso Travel Story

**Contoso Travel** is a mid-size travel agency that has been matching customers with dream vacations for over a decade. Their team of travel advisors handles thousands of inquiries every week — finding flights, booking hotels, arranging car rentals — across destinations worldwide. But the business is scaling faster than the team can keep up. Customers expect instant, personalized recommendations at all hours. Leadership has decided to build an **AI-powered travel assistant** — a system of intelligent agents that can search inventory, make recommendations, and handle complex multi-leg trip planning just like their best human advisors.

**You've been brought on as the developer to build this system.**

### 🔴 The Problem You Face Right Now

- Before you write a single line of agent code, you need the right cloud infrastructure. 
- Your AI agents will need a **model deployment** to power their reasoning, **Application Insights** to observe their behavior, and a **Foundry project** to tie it all together. 
- Without this foundation, nothing else works.

### ✅ What This Lab Solves

This lab walks you through the one-time setup:
 - creating your Microsoft Foundry project, 
 - deploying the AI model, 
 - enabling observability, and configuring the environment variables that every subsequent lab depends on. 
 
Think of it as laying the foundation before building the house.

## 🏗️ What We're Building

Throughout this workshop, we'll build a **multi-agent travel assistant** for Contoso Travel. Here's the architecture:

### The Agents

| Agent | Name | Role |
|-------|------|------|
| 🧑‍💼 **Concierge** | `contoso-travel-concierge` | The front-desk agent — greets customers, understands their travel needs, and routes requests to specialists |
| ✈️ **Flight Agent** | `contoso-flight-agent` | Searches Contoso's flight inventory by route, date, price, and cabin class |
| 🏨 **Hotel Agent** | `contoso-hotel-agent` | Finds hotels by city, star rating, price, and amenities |
| 🚗 **Car Rental Agent** | `contoso-carrental-agent` | Looks up rental cars by city, vehicle type, and availability |
| 🔀 **Workflow Orchestrator** | `contoso-travel-workflow` | Coordinates the three specialist agents into a single trip-planning workflow |

### The Data

Each specialist agent is backed by real inventory data stored as CSV files in `labs/data/contoso-travel/`:

| File | Records | Coverage |
|------|---------|----------|
| `flights.csv` | 20 flights | 5 route pairs: Seattle↔Paris, NYC↔London, SF↔Tokyo, Chicago↔Rome, Denver↔Cancún |
| `hotels.csv` | 15 hotels | 5 destination cities: Paris, London, Tokyo, Rome, Cancún |
| `car_rentals.csv` | 15 vehicles | Same 5 cities, with Economy, SUV, Luxury, and Minivan options |
| `evaluation_data.jsonl` | 10 Q&A pairs | Ground-truth queries and expected responses for evaluation |
| `sample_prompts.jsonl` | 10 prompts | Realistic customer queries for testing |

Rich markdown descriptions of each domain are also provided in `labs/data/contoso-travel/md/` for grounding and file-search tooling.

### The Journey

| Lab | What You Build |
|-----|---------------|
| **00 (this lab)** | Set up the Foundry project, deploy a model, enable observability |
| **01** | Install the SDK, validate your connection, preview the data |
| **02** | Create the Concierge — your first prompted agent |
| **03a** | Add function tools so the agent queries real flight/hotel/car data |
| **03b** | Break it into specialist agents orchestrated by a workflow |
| **04** | Add tracing — console first, then Azure Monitor |
| **05** | Evaluate quality and safety with built-in evaluators |
| **06** | Red-team the agent with adversarial attack simulations |

## 1. Prerequisites

Before starting, ensure you have:

| Requirement | Details |
|------------|---------|
| **Azure Subscription** | An active subscription with permissions to create resources. [Create a free account](https://azure.microsoft.com/free/) if needed. |
| **GitHub Account** | To fork this repo and launch GitHub Codespaces. |
| **Python & Jupyter** | Pre-installed if using GitHub Codespaces with this repo's devcontainer. |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡</strong> If you launched this repo via <b>GitHub Codespaces</b>, your dev environment is already configured — skip straight to Step 2.</div>

---


## 2. Create a Microsoft Foundry Project

‼️‼️‼️ TODO: Change this to reflect new Foundry UI ‼️‼️‼️

A **Foundry project** is your workspace for managing AI models, agents, evaluations, and traces.

### Steps:

1. Navigate to the [Microsoft Foundry portal](https://ai.azure.com)
1. Sign in with your Azure account
1. Slide the Foundry toggle to use the _New_ Foundry portal experience.
1. Click **"+ Create new project"** - and complete workflow.
  - Fill in a name of your choice for the project (e.g., `nitya-mvpsummit-demo`)
  - Select a supported region (recommended: `EastUS 2` or `Sweden Central`)
  - Other regions may not have required support for red teaming or safety evaluators.
1. Wait for the project to be provisioned (this may take 1-2 minutes)

### What gets created:
- A **Foundry project** — your workspace for agents and evaluations
- A **Foundry resource** — for managing assets across projects 

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #3f51b5; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>📸 Screenshot reference:</strong> After creation, you should see the project Overview page with your <b>Project endpoint</b> displayed.</div>

![Foundry Project Overview](https://learn.microsoft.com/en-us/azure/ai-foundry/media/how-to/projects/project-settings.png)

---


## 3. Deploy an AI Model

Our agents need an underlying language model. We'll deploy **GPT-4.1 mini** (cost-effective, fast) — but you can use GPT-4.1 or another model if preferred.

### Steps:

1. Switch to Build tab, select **Models**
2. Click **"+ Deploy model"** → **"Deploy base model"**
3. Search for and select **`gpt-4.1-mini`**  
4. Configure the deployment:
   - **Deployment name:** `gpt-4.1-mini` (keep this name — we'll reference it in our `.env`)
   - **Deployment type:** Standard
   - **Tokens per Minute Rate Limit:** Set to at least **30K** (increase if available — this affects how fast evaluations and red teaming run)
5. Click **"Deploy"**
6. Wait for the deployment status to show **"Succeeded"**

### Record these values (you'll need them later):

| Value | Where to Find | Example |
|-------|--------------|---------|
| **Deployment Name** | Models + endpoints → Name column | `gpt-4.1-mini` |
| **Model Endpoint** | Models + endpoints → click deployment → Endpoint URL | `https://contoso-travel.services.ai.azure.com` |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡 Tip:</strong> If you plan to use GPT-4.1 for higher-quality responses, deploy it as a second model. The labs use a single model deployment name via the <code>AZURE_AI_MODEL_DEPLOYMENT_NAME</code> variable.</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>⚠️ Rate Limits:</strong> Evaluations and red teaming make many API calls. If you hit rate limits, increase the TPM (Tokens Per Minute) quota on your deployment.</div>

---


## 4. Enable Application Insights for Tracing

**Application Insights** is where agent traces are sent and stored. The Foundry portal's **Tracing** tab reads from Application Insights. This is critical for Lab 04 (Tracing).

### Steps:

1. In your Foundry project, click **"Tracing"** in the left navigation
2. If prompted, click **"Enable"** to connect Application Insights
   - If Application Insights was auto-created with your project, it may already be enabled
   - If not, you'll be prompted to select or create an Application Insights resource
3. Verify the connection:
   - You should see the Tracing dashboard (it will be empty — no traces yet)
   - Note: the **Application Insights connection string** is retrieved programmatically by the SDK in Lab 04

### Verify in Azure Portal (optional):

1. Go to the [Azure portal](https://portal.azure.com)
2. Navigate to your resource group
3. Find the **Application Insights** resource
4. Confirm it exists and is in the same region as your Foundry project

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡</strong> <b>Why this matters:</b> Without Application Insights, traces from Lab 04 will not appear in the Foundry portal. Console tracing will still work, but Azure Monitor tracing requires this setup.</div>

---


## 5. Configure Environment Variables

All lab notebooks load settings from a shared `.env` file. We provide a **setup script** that auto-discovers your Azure resources and populates it for you.

### Environment Variables Reference

| Variable | Required By | Description |
|----------|-------------|-------------|
| `AZURE_AI_PROJECT_ENDPOINT` | All labs | Full project endpoint (Overview page) |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | All labs | Deployed model name (e.g., `gpt-4.1-mini`) |
| `MODEL_ENDPOINT` | Red Teaming labs | Base AI Services URL (no `/api/projects/...`) |
| `MODEL_API_KEY` | Red Teaming labs | AI Services API key |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡 Tip:</strong> The <code>AZURE_AI_PROJECT_ENDPOINT</code> includes <code>/api/projects/&lt;name&gt;</code> at the end. The <code>MODEL_ENDPOINT</code> is just the base URL (<code>https://&lt;account&gt;.services.ai.azure.com</code>). They come from the same account but are used differently.</div>

---


## 6. Run the Setup Script (Recommended)

The `setup-env.sh` script in `labs/notebooks/` will:

1. Check your Azure CLI login (and prompt `az login` if needed)
2. Create a `.env` file from the provided `sample.env` template
3. Auto-discover your AI Services account, Foundry project, model deployments, and API keys
4. Report which values were set and which still need manual entry

Run it from the terminal or from this notebook:


In [ ]:
# Run setup-env.sh to auto-discover Azure resources and populate .env
# Two cd commands: first to repo root, then into labs/notebooks/
# where setup-env.sh and sample.env template live
!cd ../.. && cd labs/notebooks && bash setup-env.sh

### Manual Setup (Alternative)

If you prefer to set values manually or the script couldn't discover everything:

1. Copy the template: `cp labs/notebooks/sample.env labs/notebooks/.env`
2. Open `.env` and fill in the values from your Foundry portal:

| Value | Where to Find |
|-------|---------------|
| `AZURE_AI_PROJECT_ENDPOINT` | Foundry Portal → Your Project → Overview → Project endpoint |
| `AZURE_AI_MODEL_DEPLOYMENT_NAME` | Foundry Portal → Models + endpoints → Name column |
| `MODEL_ENDPOINT` | Models + endpoints → click deployment → Target URI (base URL only) |
| `MODEL_API_KEY` | Models + endpoints → click deployment → Show key |

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #e91e63; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>🔒 Security:</strong> The <code>.env</code> file is git-ignored. Never commit API keys to source control.</div>

---


## 7. Authenticate with Azure

The SDK uses `DefaultAzureCredential` which tries multiple authentication methods. In GitHub Codespaces, the easiest approach is to log in via the Azure CLI.

### Run the cell below and follow the device-code login flow:

---


In [ ]:
# Device-code flow: required in Codespaces where no browser is available.
# Opens a URL + code you paste at https://microsoft.com/devicelogin
!az login --use-device-code

## 8. Validate Your Setup

Let's verify everything is configured correctly before moving on to the labs.

---


In [ ]:
# Validate that .env has all vars needed across workshop labs
import os
from dotenv import load_dotenv

# abspath(".") = notebook CWD (e.g. labs/notebooks/0-setup/)
# dirname() goes up one level → labs/notebooks/ where .env lives
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

print("🔍 Checking environment variables...\n")

# Map each var to its requirement level and which lab needs it
checks = {
    "AZURE_AI_PROJECT_ENDPOINT": {"required": True, "lab": "All labs"},
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": {"required": True, "lab": "All labs"},
    "MODEL_ENDPOINT": {"required": False, "lab": "Lab 06 (Red Teaming)"},
    "MODEL_API_KEY": {"required": False, "lab": "Lab 06 (Red Teaming)"},
}

all_required_set = True
for var, info in checks.items():
    value = os.environ.get(var)
    if value:
        # Truncate display to avoid leaking full keys/endpoints
        display = value[:15] + "..." + value[-4:] if len(value) > 25 else value
        print(f"  ✅ {var} = {display}")
    elif info["required"]:
        print(f"  ❌ {var} is NOT SET  ← Required for {info['lab']}")
        all_required_set = False
    else:
        print(f"  ⚠️  {var} is not set  ← Needed for {info['lab']}")

if all_required_set:
    print("\n🎉 All required variables are set!")
else:
    print("\n❌ Please set the missing required variables in .env before continuing.")

In [ ]:
# End-to-end validation: credential → project → model → App Insights
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

# Tries az CLI creds, managed identity, env vars, etc. in order
credential = DefaultAzureCredential()
# AIProjectClient wraps Foundry services: agents, evals, telemetry
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

print(f"✅ Connected to Foundry project!")
print(f"   Endpoint: {endpoint}")

# get_openai_client() returns an OpenAI client pre-wired to your
# Foundry endpoint + credentials — no manual URL/key setup needed
openai_client = project_client.get_openai_client()
# responses.create is the newer Responses API (not chat completions)
response = openai_client.responses.create(
    model=model_name,
    input="Say 'Contoso Travel is ready for takeoff!' in one sentence.",
)
print(f"\n✅ Model '{model_name}' is responding!")
print(f"   Response: {response.output_text}")

# App Insights is required for Lab 04 tracing to appear in the portal
try:
    conn_string = project_client.telemetry.get_application_insights_connection_string()
    if conn_string:
        print(f"\n✅ Application Insights is connected!")
        print(f"   Connection string: {conn_string[:40]}...")
    else:
        print(f"\n⚠️  Application Insights returned empty connection string.")
        print(f"   Go to Foundry portal → Tracing → Enable Application Insights")
except Exception as e:
    print(f"\n⚠️  Could not retrieve Application Insights connection string: {e}")
    print(f"   This is needed for Lab 04 (Tracing). Enable it in the Foundry portal.")

In [ ]:
# Release HTTP connections and token caches to avoid resource leaks
openai_client.close()
project_client.close()
credential.close()
print("✅ Clients closed. Setup validation complete!")

## 9. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ What We Accomplished</h3>
  <ul style="margin-bottom: 0;">
  <li>Created a Microsoft Foundry project with all required Azure resources</li>
  <li>Configured environment variables in a <code>.env</code> file</li>
  <li>Validated the complete setup end-to-end</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ Next:</strong> Head to <code>1-prompt-agents/</code> and start with <a href=\"../1-prompt-agents/lab-01-setup.ipynb\">Lab 01</a> to install SDK dependencies and explore the Contoso Travel sample data!\n
</div>

---
